In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<img src="./images/nvmath_head_panel@0.5x.png" alt="nvmath-python" />

# Getting Started with nvmath-python: Multi-GPU Multi-Node (MGMN) Execution

## Overview

This notebook provides an introduction to the *distributed host APIs* in **nvmath-python**, with which you can solve large problems in parallel across multiple GPUs and nodes. The focus is on understanding how to start the distributed runtime on multiple processes, and how to perform distributed math operations. For an in-depth explanation of these APIs, please refer to the [Distributed Host APIs](https://docs.nvidia.com/cuda/nvmath-python/latest/distributed-apis/) documentation.

**Learning Objectives:**
* Understand the single-GPU execution space (using `nvmath.fft.fft()`).
* Understand the change in execution space from single GPU to MGMN.
* Use `nvmath.distributed.fft.fft()` to perform distributed FFT operations.
* (optional) Exercise at the end to complete a distributed matrix multiplication example.

---
## Introduction

In the [second notebook](./02_mem_exec_spaces.ipynb)
of the tutorial, you learned about execution and memory spaces. This notebook is an introduction to *executing math operations in parallel across Multiple GPUs
and Multiple Nodes (MGMN)* with nvmath-python.

The **nvmath-python distributed APIs** are backed by NVIDIA high-performance multi-process libraries, such as
[cuFFTMp](https://docs.nvidia.com/cuda/cufftmp/),
[cuBLASMp](https://docs.nvidia.com/cuda/cublasmp/),
and [cuSOLVERMp](https://docs.nvidia.com/cuda/cusolvermp/).
The APIs provide Pythonic access to all the features and performance of these libraries.

This notebook demonstrates the use of **nvmath-python** distributed APIs, walking through simple examples that show how to go from performing Fast Fourier Transform (FFT) on a single GPU to performing distributed FFT on multiple GPUs.
The distributed APIs follow the "Single Program, Multiple Data" (SPMD) parallel programming model, where multiple processes execute the same program simultaneously, but each process has a different subset of the data.

**Prerequisites:** To use this notebook, you will need:
- A computer equipped with two or more NVIDIA GPUs ([Brev](https://developer.nvidia.com/brev) can be used if you don't have this readily available).
- An environment with properly installed Python libraries as described in [01_kernel_fusion.ipynb](./01_kernel_fusion.ipynb) and the Setup section below.
- (recommended) Completion of the previous notebooks on the NVIDIA Accelerated Computing Tutorial for **nvmath-python**.
- Understanding of basic GPU and parallel computing concepts (such as SPMD model).

---
## Setup

This notebook requires the following Python libraries:
- `nvmath-python`: NVIDIA's mathematical library for Python.
- `cupy`: For GPU array operations.
- `mpi4py`: To use MPI as bootstrapping mechanism for nvmath.distributed.
- `ipyparallel`: To launch multiple processes from the notebook using MPI, and execute code on them interactively.
- `torch` (optional): For PyTorch tensor operations, and to use torch.distributed as optional bootstrapping mechanism.
- `nbdistributed` (optional): To launch multiple processes from the notebook using torch.distributed, and execute code on them interactively.

If you completed the previous notebooks, the additional packages to install are:

```bash
pip install nvmath-python[cu13-distributed] mpi4py openmpi ipyparallel nbdistributed
```

**Note:** mpi4py requires an MPI implementation. There are convenient binary pip packages for [openmpi](https://pypi.org/project/openmpi/) and [mpich](https://pypi.org/project/mpich/) which mpi4py automatically recognizes if installed.

For detailed installation instructions, please refer to the [nvmath-python
documentation](https://docs.nvidia.com/cuda/nvmath-python/latest/installation.html#install-nvmath-python).

---
## Computing a 2D FFT on a single GPU

Let's start with what we learned in [previous tutorials](./02_mem_exec_spaces.ipynb): how to compute an FFT with `nvmath.fft` on a single GPU.

The following computes a complex-to-complex (C2C) 2D forward FFT:

In [ ]:
import cupy as cp
import nvmath

# Create a 2D complex array on the GPU (CuPy)
shape = 1024, 1024
a = cp.random.rand(*shape) + 1j * cp.random.rand(*shape)  # complex128 by default

In [ ]:
# Perform the 2D forward FFT using nvmath.fft
result = nvmath.fft.fft(a)

print(f"Input  shape: {a.shape}, dtype: {a.dtype}, device: GPU")
print(f"Output shape: {result.shape}, dtype: {result.dtype}, device: GPU")
print(result)

<hr style="height: 3px; background-color: black; border: none;">

## From single-GPU to multi-GPU multi-node execution

Now let's walk through using the `nvmath.distributed` APIs to scale the above code to MGMN execution.

The APIs to perform distributed math operations closely mirror the single-GPU APIs, with **a few key differences**:

- The application is launched with multiple processes. Both MPI (e.g. with mpiexec) and `torch.distributed` (e.g. with torchrun) are supported.
- There is one process per GPU.
- The operands on each process are the local partition of the global operands (as in the *Single program, multiple data -SPMD- model*) and the user specifies the distribution (how the data is partitioned across processes).

We'll cover each of these aspects below.

This notebook is designed to be run on a single node with two GPUs. We'll demonstrate how to initialize
`nvmath.distributed` using both MPI and `torch.distributed`. For the purposes of this tutorial
and to facilitate launching and managing multiple MPI or torch.distributed processes interactively from the
notebook, we'll be using the ipyparallel and nbdistributed packages. On production systems, please consult
your cluster's documentation for how to launch multi-process multi-node jobs with MPI or PyTorch.

---
### Launching multiple processes with MPI

The following instruction registers the `%%mgmn` magic which we'll use throughout the notebook to execute code in parallel across all the worker processes.

In [ ]:
%run _mgmn_nb_setup.py

Now we launch two worker processes using MPI (<u>this requires that there are at least two GPUs on this node</u>).
The workers will run nvmath-python distributed APIs.

In [ ]:
import ipyparallel as ipp

# Initialize mpi4py worker processes.
cluster = ipp.Cluster(engines="mpi", n=2)
client = cluster.start_and_connect_sync()
client.activate()

Let's verify that the workers are running and ready for code execution. **Note the use of the `%%mgmn` magic to run the code on the workers.**

In [ ]:
%%mgmn

print("hello world")

The above output shows the output of each process.

---
### Initializing the `nvmath.distributed` runtime

Once the processes are launched, the next step is to initialize `nvmath.distributed` on every process.

As mentioned above, you can use either MPI or `torch.distributed` to bootstrap `nvmath.distributed`.
It's important to note that this is just for bootstrapping and is *not* the communication
backend that will be used for compute. Instead, the underlying math library (such as cuFFTMp or cuBLASMp)
directly use NCCL or NVSHMEM.

To initialize the runtime, you specify to which device the process is assigned to,
the process group that will participate in distributed API calls, and the communication backends to use
for compute (NCCL and/or NVSHMEM).

Let's initialize the runtime on every worker (reminder that we're using `%%mgmn` to run code on the workers):

In [ ]:
%%mgmn

# Use the MPI bootstrapping scheme, since the workers were launched with MPI.

from cuda.core import system
from mpi4py import MPI

import nvmath.distributed
from nvmath.distributed import MPIProcessGroup

# Use the MPI communicator to define the process group.
process_group = MPIProcessGroup(MPI.COMM_WORLD)

# Get the device ID for this process (based on the rank of this process in the group)
device_id = process_group.rank % system.get_num_devices()

# Initialize nvmath.distributed.
# Distributed FFT requires the NVSHMEM communication backend.
nvmath.distributed.initialize(device_id, process_group, backends=["nvshmem"])
print(f"nvmath.distributed initialized on device {device_id}")

---
### Operand distribution

Before calling distributed math operations, the operand's data must be distributed among processes/GPUs in a way that the operation in question supports. Generally speaking, the process by which the operand's data is distributed depends on where that data is coming from (e.g. upstream operations, I/O, etc.) and is outside the scope of the math operation. **The operation only needs to know how the data is partitioned.**

As a very simple example, consider a global 1D array of only 4 elements: `np.array([0, 1, 2, 3])`. One way to distribute this array on two processes is to have the first process hold the first two elements and the second process the last two:

In [ ]:
%%mgmn

import numpy as np

# Get this process' rank in the process group.
rank = process_group.rank

if rank == 0:
    data = np.array([0,1])
elif rank == 1:
    data = np.array([2,3])

# The data is distributed.
print(data)

A simple and efficient way to distribute an n-dimensional array for FFT is to partition the array on a single axis. This is referred to as the **Slab** distribution in nvmath-python, and is illustrated in the following figures:

<p float="left">
  <img src="./images/slab-x.png"/>
  <img src="./images/slab-y.png"/>
</p>

nvmath-python supports several other distributions types (including **Block Cyclic**). Refer to the [documentation](https://docs.nvidia.com/cuda/nvmath-python/latest/distributed-apis/distribution.html)   for more information.

The previous 1D array conforms to a `Slab.X` distribution, since it is uniformly partitioned on its single dimension:

In [ ]:
%%mgmn

from nvmath.distributed.distribution import Slab

# data conforms to Slab.X
distribution = Slab.X

# Verify that the local shape of data conforms to Slab.X
global_shape = (4,)  # global 1D array of length 4
# Get the local shape on this rank according to the distribution.
local_shape = distribution.shape(rank, global_shape)
print(f"Local shape according to {distribution} is {local_shape}")

assert local_shape == data.shape

---
### Distributed FFT

Now let's see how to perform a distributed FFT. As we'll see, the code is similar to the single-GPU version.

**It's important to note that the code is independent of the bootstrapping mechanism used.**

We'll use the same global problem size as before (2D array of size 1024x1024). This time, the array
is distributed across multiple processes using the `Slab.X` distribution (we saw an example of this above).

One important consideration is that the underlying library (cuFFTMp) uses NVSHMEM for communication
and requires the GPU operand to be on NVSHMEM symmetric heap. nvmath-python provides convenient helpers
to allocate CuPy arrays or PyTorch tensors on symmetric memory.

In [ ]:
%%mgmn

import cupy as cp
from nvmath.distributed.distribution import Slab

# Global shape: (1024, 1024), partitioned along the X axis across ranks.
global_shape = 1024, 1024
local_shape = Slab.X.shape(process_group.rank, global_shape)

# cuFFTMp requires GPU operands to be on NVSHMEM symmetric memory.
a = nvmath.distributed.allocate_symmetric_memory(local_shape, cp, dtype=cp.complex128)
with cp.cuda.Device(device_id):
    a[:] = cp.random.rand(*local_shape) + 1j * cp.random.rand(*local_shape)

After this step, the distributed input array has been allocated and populated on each process.

Let's verify the shape and dtype of the operand on each process.

In [ ]:
%%mgmn

a.shape, a.dtype

Now we can call the `nvmath.distributed.fft.fft()` function to compute the FFT using multiple GPUs.

In [ ]:
%%mgmn

# --- Distributed forward FFT (with input distributed according to Slab.X distribution) ---
result = nvmath.distributed.fft.fft(a, distribution=Slab.X)

print(f"Local result shape: {result.shape}, dtype: {result.dtype}")

**IMPORTANT**: Operands on symmetric memory owned by the user have to be freed explicitly. See [nvmath-python symmetric memory management](https://docs.nvidia.com/cuda/nvmath-python/0.9.0/distributed-apis/utils.html#nvshmem-symmetric-memory-management) for more details on this requirement.

In [ ]:
%%mgmn

nvmath.distributed.free_symmetric_memory(a)

---
### Shutting down the workers

Finally, let's shut down the MPI workers:

In [ ]:
client.shutdown()

<hr style="height: 3px; background-color: black; border: none;">

## Exercise: Distributed Matrix Multiplication

As an exercise, write an example that does a matrix multiplication using the distributed APIs (for multi-GPU execution).

### Launch multiple processes with torch.distributed

For this exercise you can optionally launch the workers with `torch.distributed` instead of MPI. If you don't have PyTorch installed, feel free to launch workers with MPI following [From single-GPU to multi-GPU multi-node execution](#From-single-GPU-to-multi-GPU-multi-node-execution) and once the workers are running and `nvmath.distributed` is initialized go down to [Distributed matrix multiplication](#Distributed-matrix-multiplication) to run your code.

For the purposes of this tutorial we'll use nbdistributed to launch `torch.distributed`-enabled workers (in production environments you might use torchrun instead).
The following instructions load the nbdistributed notebook extension and register the `%%mgmn` magic which we
use to execute code in parallel across multiple worker processes.

In [ ]:
%load_ext nbdistributed
%run _mgmn_nb_setup.py

Now we can launch multiple processes using nbdistributed:

In [ ]:
# Initialize torch.distributed worker processes.
%dist_init --num-processes 2 --gpu-ids 0,1

# Disable distributed mode on cells without magic (instead we'll use the
# explicit %%mgmn magic to dispatch to workers).
%dist_mode --disable

---
Let's check the status of our small GPU cluster:

In [ ]:
%dist_status

Finally, let's verify that the workers are ready for code execution. **Note the use of the `%%mgmn` magic to run the code on the workers.**

In [ ]:
%%mgmn

print(f"Hello world. My device is {device}")

---
### Initialize nvmath.distributed runtime

This time we'll initialize the runtime using the TorchProcessGroup. A reminder that the bootstrapping mechanism (torch.distributed in this case) is only used for bootstrapping
and is *not* the communication backend that will be used for compute by nvmath-python. Instead,
the underlying math library (such as cuFFTMp or cuBLASMp) directly uses NCCL or NVSHMEM.

In [ ]:
%%mgmn

import torch

import nvmath.distributed
from nvmath.distributed.process_group import TorchProcessGroup

process_group = TorchProcessGroup(device_id=device)
# Distributed matrix multiplication requires the NCCL backend.
nvmath.distributed.initialize(device, process_group, backends=["nccl"])
print(f"nvmath.distributed initialized on device {device}")

---
### Distributed matrix multiplication

We're ready to call distributed APIs. For distributed matrix multiplication it's important to consider how the matrices are distributed, as it has a direct impact on the parallel algorithm used by cuBLASMp.

Let's assume the following case: [cuBLASMp documentation](https://docs.nvidia.com/cuda/cublasmp/usage/tp.html) explains that if the matrix multiplication layout is TN and A and B are partitioned column-wise and the result is partitioned row-wise, it will perform the operation using the AllGather+GEMM algorithm.

TN operation layout means that the global shape of A is `(K, M)` (A is transposed) and the global shape of B is `(K, N)`. Row-wise and column-wise partitioning is equivalent to Slab on X and Y, respectively.

Consider how to express this with nvmath-python distributed APIs:

In [ ]:
%%mgmn

import cupy as cp
import nvmath.distributed
from nvmath.distributed.distribution import ...

# Global M, N, K
M, K, N = 4096, 1024, 512

a_distribution = ...
b_distribution = ...
result_distribution = ...
# Distributed matmul API needs to know how the operands are distributed.
distributions = [a_distribution, b_distribution, result_distribution]

a_local_shape = ...
b_local_shape = b_distribution.shape(process_group.rank, (K, N))

# Allocate input matrices A and B.
# cuBLASMp uses NCCL and doesn't require operands on NVSHMEM symmetric heap.
with cp.cuda.Device(device):
    a = cp.random.randn(*a_local_shape).astype(cp.float32)
    b = ...

# cuBLASMp requires input matrices with Fortran (column-major) memory layout.
# For simplicity, we copy them to Fortran-ordered arrays.
a = cp.asfortranarray(a)
b = cp.asfortranarray(b)

All that is left is to call the distributed Matmul API with the operands, the TN qualifier, and the distribution of the operands:

In [ ]:
%%mgmn

import numpy as np
from nvmath.distributed.linalg.advanced import matrix_qualifiers_dtype

qualifiers = np.zeros((3,), dtype=matrix_qualifiers_dtype)
qualifiers[0]["is_transpose"] = True  # A is transposed

with nvmath.distributed.linalg.advanced.Matmul(
    a,
    b,
    distributions=distributions,
    qualifiers=qualifiers,
) as mm:
    ...
    result = ...

print(f"Local result shape: {result.shape}, dtype: {result.dtype}")

---
### Shut down the workers

Now we shut down the workers (launched with nbdistributed):

In [ ]:
%dist_shutdown

<hr style="height: 3px; background-color: black; border: none;">

## Conclusion

In this notebook, we learned the fundamentals of using **nvmath-python distributed APIs** to solve math problems at scale across multiple GPUs and nodes (MGMN), and the differences between the single-GPU execution space and MGMN execution. The distributed APIs follow the "Single Program, Multiple Data" (SPMD) parallel programming model, where multiple processes execute the same program simultaneously, but each process has a different subset of the data.

**Key Takeaways:**
- To use nvmath-python distributed APIs launch your application with multiple processes, one process per GPU.
- You can launch your application with either MPI or `torch.distributed`. This is used as a bootstrapping mechanism and not used for compute.
- Prior to calling distributed math operations, the input operands have to be distributed across processes, following the SPMD model of parallel programming.
- The APIs for distributed math operations are similar to their single-GPU counterparts. The main difference being that on each process you pass the local partition of the operand, and specify how the operand is distributed.
- nvmath-python provides several distribution types (Slab, Block-cyclic, etc.) that make it easy to work with the distributed math operation APIs: distributed FFT and dense linear algebra (matrix multiplication and direct solver).

---
## References

- NVIDIA, "nvmath-python Distributed Host APIs", https://docs.nvidia.com/cuda/nvmath-python/latest/distributed-apis/, Accessed: July 10, 2026.
- NVIDIA, "cuFFTMp Library User Guide", https://docs.nvidia.com/cuda/cufftmp/, Accessed: July 10, 2026.
- NVIDIA, "cuBLASMp Library User Guide", https://docs.nvidia.com/cuda/cublasmp/, Accessed: July 10, 2026.
- Peter Pacheco, "An Introduction to Parallel Programming", https://www.ime.usp.br/~alvaroma/ucsp/parallel/an_introduction_to_parallel_programming_-_peter_s._pacheco.pdf, Accessed: July 10, 2026
- Victor Eijkhout, "Parallel Programming in MPI and OpenMP, The Art of HPC, volume 2", https://www.cse.iitd.ac.in/~rijurekha/col380_2024/mpi_book.pdf, Accessed: July 10, 2026.
- Lisandro Dalcin, "MPI for Python (mpi4py)", https://mpi4py.readthedocs.io/, Accessed: July 10, 2026.